# Hotel Booking Agent - Tool Calling + SQLite (Gradio)

### BUSINESS CHALLENGE:

Build a chatbot that helps customers find and book hotels. Instead of the LLM making up
hotel names, prices, or availability, it will call **tools** (real Python functions) that
query a **SQLite database** of actual hotels - and only answer using what the database
returns.

This notebook explores:
- **Tool calling** - letting the LLM decide *when* and *which* function to call, and with
  what arguments, based on the customer's message
- **SQLite** - a simple local database to store hotels and bookings
- **Multiple tool calls in one turn** - e.g. "compare the Paris hotels" should trigger a
  search *and* a comparison, or two separate lookups
- **Vague customer prompts** - "I want something cheap in Paris" vs "book me something
  fancy in Tokyo for 3 nights" - the agent has to figure out what the customer actually
  means (budget/reasonable/luxurious) without being told explicitly

I have kept here also using Groq's free-tier model and Gradio's `ChatInterface`, same as Week 2.

In [1]:
import os
import sqlite3
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

c:\Users\Mustafa Ansari\Downloads\llm engineering\llm-engineering\.venv\Lib\site-packages\huggingface_hub\constants.py:298: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


#### Step 1: Load the Groq API key

Same setup as Week 2 - reads `GROQ_API_KEY` from `.env`.

In [2]:
load_dotenv(override=True)
groq_api_key = os.getenv("GROQ_API_KEY")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:8]}")
else:
    print("Groq API Key not set - check your .env file")

groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
MODEL = "llama-3.3-70b-versatile"

Groq API Key exists and begins gsk_ePDh


#### Step 2: Create the SQLite database

Here i created two tables:
- `hotels` - our hotel inventory (name, city, category, price, rating, amenities)
- `bookings` - records of reservations customers make

`db_path` points to a local file called `hotels.db`. If it already exists from a previous
run, we drop the tables first so this cell can be re-run cleanly.

In [3]:
db_path = "hotels.db"

conn = sqlite3.connect(db_path, check_same_thread=False)
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS bookings")
cursor.execute("DROP TABLE IF EXISTS hotels")

cursor.execute("""
CREATE TABLE hotels (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    city TEXT NOT NULL,
    category TEXT NOT NULL,       -- 'budget', 'reasonable', or 'luxurious'
    price_per_night REAL NOT NULL,
    rating REAL NOT NULL,         -- out of 5
    amenities TEXT NOT NULL       -- comma-separated
)
""")

cursor.execute("""
CREATE TABLE bookings (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    hotel_id INTEGER NOT NULL,
    customer_name TEXT NOT NULL,
    nights INTEGER NOT NULL,
    total_price REAL NOT NULL,
    FOREIGN KEY (hotel_id) REFERENCES hotels (id)
)
""")

conn.commit()
print("Database and tables created")

Database and tables created


### Step 3: Add sample hotels

A handful of hotels across a few cities, spread across `budget`, `reasonable`, and
`luxurious` categories, so the agent has real options to search through and compare.

In [4]:
sample_hotels = [
    ("Paris Budget Inn", "Paris", "budget", 65.0, 3.4, "wifi, breakfast"),
    ("Hotel Montmartre Comfort", "Paris", "reasonable", 140.0, 4.1, "wifi, breakfast, gym"),
    ("Le Grand Palais Paris", "Paris", "luxurious", 480.0, 4.8, "wifi, spa, pool, room service, gym"),

    ("Rome Traveler Hostel", "Rome", "budget", 55.0, 3.2, "wifi"),
    ("Hotel Colosseo Comfort", "Rome", "reasonable", 130.0, 4.0, "wifi, breakfast, gym"),
    ("Roma Luxury Suites", "Rome", "luxurious", 420.0, 4.7, "wifi, spa, pool, room service"),

    ("Tokyo Capsule Stay", "Tokyo", "budget", 45.0, 3.5, "wifi"),
    ("Shibuya Comfort Hotel", "Tokyo", "reasonable", 150.0, 4.2, "wifi, breakfast, gym"),
    ("Tokyo Imperial Luxury", "Tokyo", "luxurious", 520.0, 4.9, "wifi, spa, pool, room service, gym"),

    ("NYC Budget Stay", "New York", "budget", 90.0, 3.3, "wifi"),
    ("Manhattan Comfort Inn", "New York", "reasonable", 220.0, 4.0, "wifi, breakfast, gym"),
    ("The Grand Manhattan", "New York", "luxurious", 650.0, 4.8, "wifi, spa, pool, room service, gym"),
]

cursor.executemany(
    "INSERT INTO hotels (name, city, category, price_per_night, rating, amenities) VALUES (?, ?, ?, ?, ?, ?)",
    sample_hotels,
)
conn.commit()
print(f"Inserted {len(sample_hotels)} hotels")

Inserted 12 hotels


#### Step 4: The tool functions

These are plain Python functions that query the database. The LLM never touches SQLite
directly - it will ask us to run one of these functions, and we return the result back
to it as text.

- `search_hotels` - find hotels by city and/or category and/or max price
- `get_hotel_details` - full info on one hotel by id
- `compare_hotels` - side-by-side comparison of two or more hotels
- `book_hotel` - create a booking and return a confirmation with the total price

In [5]:
def search_hotels(city: str = None, category: str = None, max_price: float = None) -> str:
    query = "SELECT id, name, city, category, price_per_night, rating, amenities FROM hotels WHERE 1=1"
    params = []

    if city:
        query += " AND city LIKE ?"
        params.append(f"%{city}%")
    if category:
        query += " AND category = ?"
        params.append(category)
    if max_price:
        query += " AND price_per_night <= ?"
        params.append(max_price)

    cursor.execute(query, params)
    rows = cursor.fetchall()

    if not rows:
        return "No hotels found matching those criteria."

    results = []
    for row in rows:
        results.append(
            f"ID {row[0]}: {row[1]} in {row[2]} ({row[3]}) - ${row[4]}/night, "
            f"rating {row[5]}/5, amenities: {row[6]}"
        )
    return "\n".join(results)


def get_hotel_details(hotel_id: int) -> str:
    cursor.execute("SELECT id, name, city, category, price_per_night, rating, amenities FROM hotels WHERE id = ?", (hotel_id,))
    row = cursor.fetchone()

    if not row:
        return f"No hotel found with ID {hotel_id}."

    return (
        f"ID {row[0]}: {row[1]} in {row[2]} ({row[3]}) - ${row[4]}/night, "
        f"rating {row[5]}/5, amenities: {row[6]}"
    )


def compare_hotels(hotel_ids: list[int]) -> str:
    comparisons = [get_hotel_details(hid) for hid in hotel_ids]
    return "\n".join(comparisons)


def book_hotel(hotel_id: int, customer_name: str, nights: int) -> str:
    cursor.execute("SELECT name, price_per_night FROM hotels WHERE id = ?", (hotel_id,))
    row = cursor.fetchone()

    if not row:
        return f"No hotel found with ID {hotel_id}, booking failed."

    hotel_name, price_per_night = row
    total_price = price_per_night * nights

    cursor.execute(
        "INSERT INTO bookings (hotel_id, customer_name, nights, total_price) VALUES (?, ?, ?, ?)",
        (hotel_id, customer_name, nights, total_price),
    )
    conn.commit()

    return (
        f"Booking confirmed for {customer_name}: {hotel_name} for {nights} night(s), "
        f"total price ${total_price:.2f}."
    )

#### Step 5: Describe the tools to the LLM

The LLM can only call a function if I describe it in a schema - name, description, and
what arguments it takes. This is what lets the model decide *on its own* whether to call
`search_hotels`, `compare_hotels`, or `book_hotel` based on what the customer says.

In [6]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_hotels",
            "description": "Search for hotels by city, category, and/or maximum price per night. Use this whenever a customer asks about hotels in a city, or wants something cheap/reasonable/luxurious.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name, e.g. Paris"},
                    "category": {
                        "type": "string",
                        "enum": ["budget", "reasonable", "luxurious"],
                        "description": "Hotel category - use 'budget' for cheap, 'reasonable' for mid-range, 'luxurious' for fancy/expensive requests",
                    },
                    "max_price": {"type": "number", "description": "Maximum price per night in USD"},
                },
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_hotel_details",
            "description": "Get full details for a single hotel by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "hotel_id": {"type": "integer", "description": "The hotel's ID"},
                },
                "required": ["hotel_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "compare_hotels",
            "description": "Compare two or more hotels side by side, given their IDs. Use this when a customer asks to compare hotels or wants to see multiple options at once.",
            "parameters": {
                "type": "object",
                "properties": {
                    "hotel_ids": {
                        "type": "array",
                        "items": {"type": "integer"},
                        "description": "List of hotel IDs to compare",
                    },
                },
                "required": ["hotel_ids"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "book_hotel",
            "description": "Book a hotel for a customer. Only call this once the customer has clearly chosen a specific hotel and confirmed they want to book it.",
            "parameters": {
                "type": "object",
                "properties": {
                    "hotel_id": {"type": "integer", "description": "The hotel's ID"},
                    "customer_name": {"type": "string", "description": "Name of the customer making the booking"},
                    "nights": {"type": "integer", "description": "Number of nights to book"},
                },
                "required": ["hotel_id", "customer_name", "nights"],
            },
        },
    },
]

available_functions = {
    "search_hotels": search_hotels,
    "get_hotel_details": get_hotel_details,
    "compare_hotels": compare_hotels,
    "book_hotel": book_hotel,
}

#### Step 6: The system prompt

Here I tell the agent about the categories (so it maps "cheap" -> budget, "fancy" ->
luxurious, etc. on its own) and remind it to only state facts that came from a tool call.

In [20]:
system_prompt = """
You are a helpful hotel booking assistant. Keep replies short - 1 to 3 sentences.

TOOLS
Always use the tools (search_hotels, get_hotel_details, compare_hotels, book_hotel) to get
real information. Never invent hotel names, prices, ratings, or availability - if you
haven't gotten it from a tool call, don't say it.

CATEGORIES
- "budget" = cheap / affordable / low-cost
- "reasonable" = mid-range / decent / reasonably priced
- "luxurious" = fancy / expensive / high-end / premium

VAGUE REQUESTS
If a customer names a city but gives no sense of budget or category (e.g. "something nice
in Paris"), ask ONE short clarifying question about budget/category before searching.
Don't guess and don't search yet.

If a customer gives a city AND a budget signal (even an implicit one like "cheap" or
"fancy"), map it to a category yourself and search right away - don't ask a clarifying
question in that case.

CITY OR HOTEL NOT FOUND
If search_hotels returns nothing, do apologize politely and suggest alternatives by asking them first. if they said yes then, call search_hotels
again with only the category (no city) to find what cities/hotels ARE available in that
category, and offer those as alternatives. Only say "we're not available there" after
confirming there's truly nothing close to offer - never invent a city or hotel that
didn't come back from a tool call.

COMPARISONS
If a customer wants to compare two or more hotels (by name, city, or "show me a few
options"), call compare_hotels with all relevant hotel IDs and summarize the differences
in 1-2 sentences - don't just dump the raw tool output.

BOOKING
Only call book_hotel once the customer has clearly named one specific hotel, the number
of nights, and confirmed they want to book it. If any of those three are missing, ask for
the missing piece instead of booking. After booking, confirm the hotel name, nights, and
total price in one sentence.
"""

#### Step 7: The tool-calling chat function

This is the core loop:

1. Send the conversation (system prompt + history + new message) to Groq, along with the
   list of tools
2. If the model responds with one or more `tool_calls`, we run each one against our real
   Python functions and feed the results back to the model
3. We repeat this until the model responds with a normal text answer (no more tool calls)

This is what lets a single customer message trigger **multiple** tool calls - e.g. asking
to compare two hotels in different cities might trigger two `search_hotels` calls plus a
`compare_hotels` call, all before the customer sees a reply.

In [8]:
def chat(message, history):
    cleaned_history = [
        {"role": entry["role"], "content": entry["content"]}
        for entry in history
    ]
    messages = [{"role": "system", "content": system_prompt}] + cleaned_history + [{"role": "user", "content": message}]

    while True:
        response = groq.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
            temperature=0.3,
        )
        reply = response.choices[0].message

        if not reply.tool_calls:
            return reply.content

        messages.append(reply)

        for tool_call in reply.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            function = available_functions[function_name]

            result = function(**function_args)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            })

#### Step 8: Launch the Gradio chatbot
Try some scenarios with different levels of clarity


In [19]:
gr.ChatInterface(
    fn=chat,
    title="Hotel Booking Agent",
).launch()

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.
